# Envío 19 — Ensamble grande + asignación húngara

Semana 2. Hipótesis: Combina todas las mejoras.

CV local (GroupKFold por prompt, 5 folds): **0.5086** balanced accuracy.

Notebook autocontenido: celda 1 = librería común, celda 2 = definición de los experimentos, celda 3 = ejecución. Parte de la idea de los notebooks públicos heliosli/name-that-llm-lv681-a15 y omkarpattanaik123/name-that-llm-v28-sovereign-titan-apex; el aporte propio son el centrado por prompt, la asignación húngara y los ensambles descritos en el informe.

In [ ]:
"""Librería común para la competencia Name That LLM.

Contiene: carga de datos, características estilométricas, constructores de
representaciones, modelos base (todos devuelven probabilidades), utilidades de
ensamblaje y el post-procesado por clusters de prompt (asignación húngara).
Este archivo se incrusta tal cual dentro de cada notebook de Kaggle.
"""
import os, re, warnings
import numpy as np, pandas as pd
from scipy import sparse
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.cluster import AgglomerativeClustering

warnings.filterwarnings("ignore")
SEED = 42


# ----------------------------------------------------------------- datos
def find_dir():
    for d in [os.environ.get("DATA_DIR", "."), "/kaggle/input/name-that-llm", "/kaggle/input/competitions/name-that-llm", ".."]:
        if os.path.exists(os.path.join(d, "train.csv")):
            return d
    for root, _, files in os.walk("/kaggle/input"):
        if "train.csv" in files:
            return root
    raise FileNotFoundError("train.csv no encontrado")


def load():
    d = find_dir()
    return pd.read_csv(f"{d}/train.csv"), pd.read_csv(f"{d}/test.csv")


def write_submission(test, labels, path="submission.parquet"):
    sub = pd.DataFrame({"id": test["id"].values, "model": np.asarray(labels)})
    assert sub["model"].notna().all() and len(sub) == len(test)
    sub.to_parquet(path, index=False)
    return sub


# ------------------------------------------------------- estilometría
_PUNCT = list(",.;:!?—–-\"'“”‘’()[]…*#/%$&_`") + ["\n"]
_FUNC = ("the of and to a in that is it for as with was on be by this which "
         "not but or are from at an have has more than their its we our can "
         "however thus yet while rather").split()


def stylo_one(t):
    n = max(len(t), 1)
    words = re.findall(r"[A-Za-zÀ-ÿ']+", t)
    nw = max(len(words), 1)
    sents = [s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s]
    sl = np.array([len(s.split()) for s in sents]) if sents else np.array([0])
    wl = np.array([len(w) for w in words]) if words else np.array([0])
    low = [w.lower() for w in words]
    paras = [p for p in t.split("\n") if p.strip()]
    f = [
        np.log1p(n), np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(),
        wl.mean(), wl.std(), len(set(low)) / nw,
        sum(c.isupper() for c in t) / n, sum(c.isdigit() for c in t) / n,
        sum(ord(c) > 127 for c in t) / n, sum(c == " " for c in t) / n,
        len(paras), t.count("\n\n"), len(re.findall(r"\b\w+n't\b|\b\w+'(s|re|ve|ll|d|m)\b", t)) / nw,
        len(re.findall(r"^\s*([-*•]|\d+[.)])\s", t, flags=re.M)),
        sum(len(w) > 9 for w in words) / nw,
        len(re.findall(r"\b[A-Z][a-z]+\b", t)) / nw,
    ]
    f += [t.count(p) / n * 100 for p in _PUNCT]
    cnt = {}
    for w in low:
        cnt[w] = cnt.get(w, 0) + 1
    f += [cnt.get(w, 0) / nw * 100 for w in _FUNC]
    return f


def stylo_matrix(texts):
    return np.array([stylo_one(t) for t in texts], dtype=float)


# ----------------------------------------------------- representaciones
def tfidf_word(ngram=(1, 2), min_df=2, lower=True, max_features=None):
    return TfidfVectorizer(ngram_range=ngram, min_df=min_df, sublinear_tf=True, lowercase=lower,
                           token_pattern=r"(?u)\b\w+\b|[^\w\s]", max_features=max_features)


def tfidf_char(ngram=(2, 5), min_df=2, lower=False, analyzer="char", max_features=None):
    return TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, min_df=min_df, sublinear_tf=True,
                           lowercase=lower, max_features=max_features)


def fit_transform_many(vecs, tr_text, te_text):
    A, B = [], []
    for v in vecs:
        A.append(v.fit_transform(tr_text)); B.append(v.transform(te_text))
    return sparse.hstack(A).tocsr(), sparse.hstack(B).tocsr()


# ------------------------------------------------------------ modelos
def softmax(z, T=1.0):
    z = z * T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def proba_of(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    return softmax(model.decision_function(X), 4.0)


def fit_predict_sparse(model, Xtr, ytr, Xte):
    model.fit(Xtr, ytr)
    return proba_of(model, Xte)


def fit_predict_stylo(model, tr_text, ytr, te_text, scale=True):
    A, B = stylo_matrix(tr_text), stylo_matrix(te_text)
    if scale:
        sc = StandardScaler().fit(A); A, B = sc.transform(A), sc.transform(B)
    model.fit(A, ytr)
    return proba_of(model, B)


def blend(ps, w=None):
    w = np.ones(len(ps)) if w is None else np.asarray(w, float)
    out = sum(wi * np.log(np.clip(p, 1e-9, 1)) for wi, p in zip(w, ps)) / w.sum()
    return softmax(out)


def blend_arith(ps, w=None):
    w = np.ones(len(ps)) if w is None else np.asarray(w, float)
    return sum(wi * p for wi, p in zip(w, ps)) / w.sum()


# ------------------------------- post-proceso: 1 texto por modelo y prompt
def cluster_prompts(texts, n_models=18, min_n=None):
    """Cada prompt tiene exactamente un pasaje por modelo, así que n/18 = nº de
    prompts. Agrupa los textos por tema (TF-IDF de contenido + SVD + clustering
    aglomerativo coseno). Con pocos textos (3 ejemplos interactivos) cada texto
    queda en su propio grupo."""
    n = len(texts)
    if n < (min_n or 2 * n_models):
        return np.arange(n)
    k = max(1, int(round(n / n_models)))
    v = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2, max_df=0.3)
    X = v.fit_transform(texts)
    Z = TruncatedSVD(min(100, X.shape[1] - 1), random_state=SEED).fit_transform(X)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9
    return AgglomerativeClustering(n_clusters=k, metric="cosine", linkage="average").fit_predict(Z)


def group_center(M, groups):
    """Resta la media del grupo (prompt) a cada fila: elimina el efecto del tema."""
    M = np.asarray(M, float); out = M.copy()
    for g in np.unique(groups):
        i = groups == g
        out[i] = M[i] - M[i].mean(0)
    return out


def group_rank(M, groups):
    """Rango percentil de cada texto dentro de su grupo, columna a columna."""
    M = np.asarray(M, float); out = np.zeros_like(M)
    for g in np.unique(groups):
        i = np.where(groups == g)[0]
        if len(i) < 2:
            out[i] = 0.5; continue
        out[i] = M[i].argsort(0).argsort(0) / (len(i) - 1)
    return out


def assign_in_groups(proba, groups, n_models):
    """Asignación húngara dentro de cada grupo: cada modelo a lo sumo una vez."""
    pred = proba.argmax(1).copy()
    lp = np.log(np.clip(proba, 1e-9, 1))
    for c in np.unique(groups):
        idx = np.where(groups == c)[0]
        if len(idx) < 2 or len(idx) > 2 * n_models:
            continue
        r, col = linear_sum_assignment(-lp[idx])
        pred[idx[r]] = col
    return pred


def prompt_assign(texts, proba, n_models, groups=None):
    g = cluster_prompts(texts, n_models) if groups is None else groups
    return assign_in_groups(proba, g, n_models)


# ------------------------------------------------------------- validación
def cv_eval(fit_predict, train, n_splits=5, post=None, seed=SEED, verbose=True):
    """GroupKFold por prompt_id. fit_predict(tr_df, te_df, classes) -> proba (n,18).
    Devuelve (balanced_accuracy, oof_proba). Si post!=None aplica el post-proceso
    por fold sobre los textos retenidos (sin usar prompt_id)."""
    le = LabelEncoder().fit(train["model"])
    y = le.transform(train["model"])
    oof = np.zeros((len(train), len(le.classes_)))
    pred = np.zeros(len(train), int)
    for f, (a, b) in enumerate(GroupKFold(n_splits).split(train, y, train["prompt_id"])):
        p = fit_predict(train.iloc[a].reset_index(drop=True), train.iloc[b].reset_index(drop=True))
        oof[b] = p
        pred[b] = post(train.iloc[b]["text"].tolist(), p, len(le.classes_)) if post else p.argmax(1)
    ba = balanced_accuracy_score(y, pred)
    return ba, oof, pred


In [ ]:
"""Los 20 experimentos. Cada uno es una función  fit_predict(tr, te) -> proba (len(te), 18)
con columnas ordenadas como LabelEncoder().fit(train.model).classes_.
EXPERIMENTS[i] = dict(id, name, week, fp, assign(bool), hyp)  — assign=True aplica el
post-proceso de clusters de prompt + asignación húngara sobre las probabilidades."""
import numpy as np
import lightgbm as lgb

_LE = None
def le_of(tr):
    return LabelEncoder().fit(tr["model"])

def _y(tr): return le_of(tr).transform(tr["model"])

def _groups(tr, te):
    return tr["prompt_id"].values, cluster_prompts(te["text"].tolist())

def _dense_centered(vecs, tr, te):
    X, Z = fit_transform_many(vecs, tr.text, te.text)
    ga, gb = _groups(tr, te)
    return group_center(X.toarray(), ga), group_center(Z.toarray(), gb)

def _stylo(tr, te, mode="center"):
    A, B = stylo_matrix(tr.text), stylo_matrix(te.text)
    ga, gb = _groups(tr, te)
    if mode == "raw": return A, B
    if mode == "center": return group_center(A, ga), group_center(B, gb)
    return (np.hstack([A, group_center(A, ga), group_rank(A, ga)]),
            np.hstack([B, group_center(B, gb), group_rank(B, gb)]))

def _scaled(A, B):
    sc = StandardScaler().fit(A); return sc.transform(A), sc.transform(B)

# ------------------------------------------------------------ SEMANA 1
def e01(tr, te):   # NB sobre TF-IDF de palabras
    X, Z = fit_transform_many([tfidf_word((1, 2))], tr.text, te.text)
    return fit_predict_sparse(MultinomialNB(alpha=0.05), X, _y(tr), Z)

def e02(tr, te):   # LR sobre TF-IDF de palabras
    X, Z = fit_transform_many([tfidf_word((1, 2))], tr.text, te.text)
    return fit_predict_sparse(LogisticRegression(C=20, max_iter=300), X, _y(tr), Z)

def e03(tr, te):   # SVM lineal sobre n-gramas de caracteres (con mayúsculas y puntuación)
    X, Z = fit_transform_many([tfidf_char((2, 5))], tr.text, te.text)
    return fit_predict_sparse(LinearSVC(C=0.5), X, _y(tr), Z)

def e04(tr, te):   # palabras + caracteres, SVM lineal
    X, Z = fit_transform_many([tfidf_word((1, 2)), tfidf_char((2, 5))], tr.text, te.text)
    return fit_predict_sparse(LinearSVC(C=0.5), X, _y(tr), Z)

def e05(tr, te):   # palabras + caracteres, regresión logística
    X, Z = fit_transform_many([tfidf_word((1, 2)), tfidf_char((2, 5))], tr.text, te.text)
    return fit_predict_sparse(LogisticRegression(C=30, max_iter=300), X, _y(tr), Z)

def e06(tr, te):   # solo estilometría (36+ rasgos) + LR
    A, B = _scaled(*_stylo(tr, te, "raw"))
    return fit_predict_sparse(LogisticRegression(C=0.3, max_iter=1000), A, _y(tr), B)

def e07(tr, te):   # TF-IDF + estilometría apilada como columnas dispersas
    X, Z = fit_transform_many([tfidf_word((1, 2)), tfidf_char((2, 5))], tr.text, te.text)
    A, B = _scaled(*_stylo(tr, te, "raw"))
    X = sparse.hstack([X, sparse.csr_matrix(A * 0.15)]).tocsr(); Z = sparse.hstack([Z, sparse.csr_matrix(B * 0.15)]).tocsr()
    return fit_predict_sparse(LogisticRegression(C=30, max_iter=300), X, _y(tr), Z)

def e08(tr, te):   # ensamble: LR(word+char) + NB + estilometría
    return blend([e05(tr, te), e01(tr, te), e06(tr, te)], [2, 1, 1])

def e09(tr, te):   # + asignación 1 texto por modelo y prompt sobre e08
    return e08(tr, te)

def e10(tr, te):   # centrado por prompt de la estilometría (LR)
    A, B = _scaled(*_stylo(tr, te, "center"))
    return fit_predict_sparse(LogisticRegression(C=0.3, max_iter=1000), A, _y(tr), B)

# ------------------------------------------------------------ SEMANA 2
def e11(tr, te):   # TF-IDF palabras+chars centrado por prompt, LinearSVC
    X, Z = _dense_centered([tfidf_word((1, 2), 3, max_features=15000), tfidf_char((2, 5), 3, max_features=25000)], tr, te)
    return fit_predict_sparse(LinearSVC(C=0.3), X, _y(tr), Z)

def e12(tr, te):   # idem con LR
    X, Z = _dense_centered([tfidf_word((1, 2), 3, max_features=15000), tfidf_char((2, 5), 3, max_features=25000)], tr, te)
    return fit_predict_sparse(LogisticRegression(C=30, max_iter=300), X, _y(tr), Z)

def e13(tr, te):   # + asignación sobre e12
    return e12(tr, te)

def e14(tr, te):   # ensamble centrado: TF-IDF centrado + estilometría centrada
    return blend([e12(tr, te), e10(tr, te)], [3, 1])

def e15(tr, te):   # estilometría completa (cruda+centrada+rango) con LightGBM
    A, B = _stylo(tr, te, "all")
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, verbose=-1, n_jobs=-1, random_state=SEED)
    m.fit(A, _y(tr)); return m.predict_proba(B)

def e16(tr, te):   # ensamble e12 + e10 + LightGBM
    return blend([e12(tr, te), e10(tr, te), e15(tr, te)], [3, 1, 1])

def e17(tr, te):   # LR centrada con n-gramas de caracteres de 1-6 y palabras 1-3
    X, Z = _dense_centered([tfidf_word((1, 3), 2, max_features=20000), tfidf_char((1, 6), 3, max_features=40000)], tr, te)
    return fit_predict_sparse(LogisticRegression(C=50, max_iter=400), X, _y(tr), Z)

def e18(tr, te):   # ensamble grande: e12+e17+e10+e15+e04(no centrado)
    return blend([e12(tr, te), e17(tr, te), e10(tr, te), e15(tr, te), e04(tr, te)], [3, 3, 1, 1, 1])

def e19(tr, te):   # e18 + asignación
    return e18(tr, te)

def e20(tr, te):   # e18 + auto-entrenamiento (pseudo-etiquetas) + asignación
    base = e18(tr, te)
    ga, gb = _groups(tr, te)
    if len(te) < 4 * 18:
        return base
    pl = assign_in_groups(base, gb, 18)
    conf = base[np.arange(len(te)), pl]
    keep = conf >= np.quantile(conf, 0.5)
    tr2 = pd.concat([tr, te[keep].assign(model=le_of(tr).classes_[pl[keep]], prompt_id=["T%d" % g for g in gb[keep]])], ignore_index=True)
    p2 = e18(tr2, te)
    return blend([base, p2], [1, 1])



In [ ]:
tr, te = load()
le = LabelEncoder().fit(tr["model"])
proba = e19(tr, te)
labels = prompt_assign(te["text"].tolist(), proba, len(le.classes_)) if True else proba.argmax(1)
sub = write_submission(te, le.classes_[labels], "submission.parquet")
print(sub.head(), sub["model"].value_counts().head())
